# SBI review + focus-param checkpoint -- Mg_Mh_s90 x Rs_Ms_s90

Manual-checkpoint workflow (see CLAUDE.md, "The shuffle test" section, for
the mean-net version of the same idea):

1. Load the already-trained noise-sweep results for this pair (all 35
   params, 7 cases, mean net) and inspect R2 heatmaps + a heuristic
   suggest_params() ranking, inline. No retraining in this section.
2. STOP -- Inspect the heatmaps above, hand-edit FOCUS_PARAMS below.
3. Train SBI (NPE-C / MAF) posteriors for just those focus params (5 cases:
   each observable alone, both clean, and both moderate-noise directions --
   no both-noisy diagonal case), then review individual-vs-combined
   posterior overlays (information sharing) and the SBI shuffle test --
   all inline, nothing saved to disk.

Auto-generated by generate_sbi_review.py. Safe to regenerate for new pairs;
FOCUS_PARAMS below is yours to edit by hand once you reach the checkpoint.

In [ ]:
import sys, os
_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_SWEEP = os.path.dirname(_HERE)
if _SWEEP not in sys.path:
    sys.path.insert(0, _SWEEP)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from _analyze_helpers import load_pair, relabel_figure, suggest_params, train_sbi_for_pair
import sbi_pipeline as sbi_pl

## 1. Load noise-sweep results (mean net, all 35 params, 7 cases)

In [ ]:
ctx = load_pair(pair_dir="pair_01_Mg_Mh_s90__Rs_Ms_s90")
ctx.summary()

## 2. R2 heatmaps -- review before picking focus params

In [ ]:
r2_df = pd.DataFrame(ctx.r2_matrix, index=ctx.cases, columns=ctx.param_names)
plt.figure(figsize=(16, 6))
sns.heatmap(r2_df, vmin=-1, vmax=1, annot=True, fmt=".2f", cmap="Spectral",
            cbar_kws={"label": "aligned R2"}, linewidths=0.2)
plt.title(f"aligned R2  |  {ctx.obs1} x {ctx.obs2}")
plt.tight_layout()
relabel_figure(ctx)
plt.show()

In [ ]:
for _name, _arr in [("shuffled (obs2 shuffled -> tests obs1)", ctx.r2_matrix_shifted_obs),
                     ("shuffled (obs1 shuffled -> tests obs2)", ctx.r2_matrix_shifted_both)]:
    _df = pd.DataFrame(_arr, index=ctx.cases, columns=ctx.param_names)
    plt.figure(figsize=(16, 6))
    sns.heatmap(_df, vmin=-1, vmax=1, annot=True, fmt=".2f", cmap="Spectral", linewidths=0.2)
    plt.title(f"{_name}  |  {ctx.obs1} x {ctx.obs2}")
    plt.tight_layout()
    relabel_figure(ctx)
    plt.show()

In [ ]:
for _name, _delta in [("delta R2 (obs2 shuffled - aligned)", ctx.r2_matrix_shifted_obs - ctx.r2_matrix),
                       ("delta R2 (obs1 shuffled - aligned)", ctx.r2_matrix_shifted_both - ctx.r2_matrix)]:
    _df = pd.DataFrame(_delta, index=ctx.cases, columns=ctx.param_names)
    plt.figure(figsize=(16, 6))
    sns.heatmap(_df, vmin=-0.5, center=0.0, annot=True, fmt=".2f", cmap="Spectral", linewidths=0.3,
                cbar_kws={"label": "delta R2 (shifted - aligned)"})
    plt.title(f"{_name}  |  {ctx.obs1} x {ctx.obs2}")
    plt.tight_layout()
    relabel_figure(ctx)
    plt.show()

## 3. Suggested params -- a heuristic starting point, not a verdict

In [ ]:
suggested = suggest_params(ctx, top_n=8)

## STOP -- CHECKPOINT

Inspect the heatmaps in section 2 above. Edit FOCUS_PARAMS in the next cell
by hand based on what you actually see, THEN continue running cells below.
`suggested["union"]` is pre-filled as a starting point only -- overwrite it
with your own picks.

In [ ]:
FOCUS_PARAMS = suggested["union"]   # <-- EDIT ME based on the heatmaps above, then continue
print("FOCUS_PARAMS =", FOCUS_PARAMS)

## 4. Train SBI (NPE-C / MAF) for the chosen focus params

5 cases -- each observable alone, both-clean, and both moderate-noise
directions (B_2.5_A_0.0 / B_0.0_A_2.5). No both-noisy diagonal case
(B_1.0_A_1.0) -- dropped, not informative enough to be worth the training
time here. A few minutes per case (NPE-C, up to 2000 epochs, early-stopped).

In [ ]:
NOISE_CASES = {
    "A_clean": {ctx.obs1: 0.0},
    "B_clean": {ctx.obs2: 0.0},
    "B_0.0_A_0.0": {ctx.obs2: 0.0, ctx.obs1: 0.0},
    "B_2.5_A_0.0": {ctx.obs2: 2.5, ctx.obs1: 0.0},
    "B_0.0_A_2.5": {ctx.obs2: 0.0, ctx.obs1: 2.5},
}
sbi_ctx = train_sbi_for_pair(ctx.obs_a, ctx.obs_b, FOCUS_PARAMS, noise_cases=NOISE_CASES,
                              loaded_data=ctx.loaded_data)
sbi_ctx.summary()
both_clean = next((r for r in sbi_ctx.all_results if r["case_name"] == "B_0.0_A_0.0"), None)

## 5. Is the flow actually fitting well?

Two checks, separate from whether it reads the right observable (that's
section 8's shuffle test):
 - Loss curves (all cases) -- cheapest sanity check, no resampling.
 - SBC rank histogram + coverage curve (both-clean case) -- the real
   calibration check. Works on raw posterior samples, not a Gaussian
   approximation, so it actually tests what the MAF is for (non-Gaussian
   posterior shape). Uniform ranks / on-diagonal coverage = calibrated.

In [ ]:
for r in sbi_ctx.all_results:
    fig = sbi_pl.plot_sbi_loss_curves(r)
    plt.show()

In [ ]:
if both_clean is not None:
    fig = sbi_pl.plot_sbc_rank_hist(both_clean, n_samples=1000, space="log_partial")
    plt.show()
    fig = sbi_pl.plot_sbc_coverage(both_clean, n_samples=1000, space="log_partial")
    plt.show()

## 6. Individual vs combined posteriors -- information sharing (clean)

Overlay observable_1-alone, observable_2-alone, and both-combined posteriors
for one test sim. If the combined posterior is meaningfully narrower than
either individual one, the two observables are sharing information for this
parameter (see CLAUDE.md's six-case observable-sharing taxonomy).

In [ ]:
SIM_IDX = 0
N_SAMPLES_CORNER = 3500   # kept below 5000 -- the k x k grid gets slow past this

_clean_cases = [r for r in sbi_ctx.all_results if r["case_name"] in ("A_clean", "B_clean", "B_0.0_A_0.0")]
_clean_labels = [sbi_ctx.display_names.get(r["case_name"], r["case_name"]) for r in _clean_cases]
fig = sbi_pl.plot_sbi_case_overlay_corner(_clean_cases, SIM_IDX, n_samples=N_SAMPLES_CORNER,
                                           space="log_partial", case_labels=_clean_labels)
plt.show()

## 7. Individual vs combined posteriors -- information sharing (moderate noise)

Same comparison, but "combined" now uses the two moderate-noise mixed cases
(one observable noised to 2.5, the other clean) instead of both-clean --
shows whether the information-sharing signature survives noise, and whether
it depends on which observable carries the noise.

In [ ]:
_noisy_cases = [r for r in sbi_ctx.all_results
                if r["case_name"] in ("A_clean", "B_clean", "B_2.5_A_0.0", "B_0.0_A_2.5")]
_noisy_labels = [sbi_ctx.display_names.get(r["case_name"], r["case_name"]) for r in _noisy_cases]
fig = sbi_pl.plot_sbi_case_overlay_corner(_noisy_cases, SIM_IDX, n_samples=N_SAMPLES_CORNER,
                                           space="log_partial", case_labels=_noisy_labels)
plt.show()

## 8. SBI shuffle test -- does the posterior actually read each observable?

For the both-clean case, shuffle one observable at inference (keep truths
in place) and check whether the posterior mean's R2 survives -- the SBI
analog of the mean-net shuffle test in CLAUDE.md. Reference cases (A_clean,
B_clean) aren't repeated here since resolve_shuffle makes them trivial
no-op/collapse pairs by construction; the both-clean case is where the
comparison is actually informative.

In [ ]:
if both_clean is not None:
    _modes = ["aligned", "obs1_vs_truth", "obs2_vs_truth"]
    _rows = []
    for _mode in _modes:
        _r2, _r2_std = sbi_pl.sbi_shuffle_r2(both_clean, _mode, n_samples=1000, space="log_partial")
        for _p, _v, _s in zip(both_clean["sbi_focus_params"], _r2, _r2_std):
            _rows.append({"mode": _mode, "param": _p, "r2": float(_v), "r2_std": float(_s)})
    shuffle_df = pd.DataFrame(_rows)
    print(shuffle_df.pivot(index="param", columns="mode", values="r2"))

In [ ]:
if both_clean is not None:
    MULTI_SIM_IDX = [0, 25, 50, 75, 101]
    for _mode in ["aligned", "obs1_vs_truth", "obs2_vs_truth"]:
        fig = sbi_pl.plot_sbi_multi_sim_corner(both_clean, MULTI_SIM_IDX, n_samples=1500,
                                                space="log_partial", mode=_mode)
        plt.show()